<a href="https://colab.research.google.com/github/syakesaba/jupyter-notebooks/blob/main/github_copilot_sdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Github Copilot SDK Python Sample Code
=====


Ref: https://github.com/github/copilot-sdk/blob/main/docs/features/index.md

# Initialize Python

In [16]:
!uv pip install --system github-copilot-sdk nest_asyncio pydantic httpx
import nest_asyncio
nest_asyncio.apply() # Google Colab自体がasyncio配下で動いているのでネストさせる。
from google.colab import userdata
GH_TOKEN=userdata.get("GH_TOKEN") # Secrets（🔑）からGH_TOKENをNotebook access可能にする
!echo {GH_TOKEN} | gh auth login --with-token

Using Python 3.12.13 environment at: /usr
Checked 4 packages in 61ms


# Github Copilot Client

In [17]:
import asyncio
from copilot import CopilotClient, SubprocessConfig

client = CopilotClient(
    SubprocessConfig(
        use_logged_in_user=False,
        github_token=GH_TOKEN,
        log_level="debug",
    )
)

asyncio.run(client.start())

# Prompting


## 最も基本的なプロンプト

In [3]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("Answer what 2+2 is."))

2+2 is 4.


## システムプロンプト

In [4]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        system_message=SystemMessageReplaceConfig(content="あなたは真逆のことを言うエージェントです")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("多くのカラスは黒いですか？"))

いいえ、多くのカラスは黒くありません。


# Tool

In [5]:
import asyncio

from copilot import CopilotSession, define_tool
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig

from pydantic import BaseModel, Field, StringConstraints

class Loto6Params(BaseModel):
    number: str = Field(description="Number for LOTO6", pattern=r'^\d{6}$')

# name must be: ^[a-zA-Z0-9_\\.-]+$
@define_tool(name="Loto6Tool", description="Loto6 check if win or not!", skip_permission=True)
async def _loto6(params: Loto6Params) -> str:
    return "WOW you got $10,000 !" if "123456" == params.number else "Sorry you got nothing!"

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        tools=[_loto6,],
        system_message=SystemMessageReplaceConfig(content="Use `Loto6Tool` if you receive 6-digits numbers and answer what you got.")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("My number is: 123456"))


Congratulations! Your number 123456 is a winner—you've won $10,000!


# Image Input

---



In [6]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

from pydantic import AnyUrl
import httpx
import base64

async def run(prompt: str, url: AnyUrl):
    async with httpx.AsyncClient() as http_client:
        image_response = await http_client.get(url)
        bin_image = image_response.content
        image_type = image_response.headers['content-type']
        b64_image = base64.b64encode(bin_image).decode('utf-8')
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(
        prompt,
        attachments=[
            {
                "type": "blob",
                "data": b64_image,
                "mimeType": image_type,
            }
        ]
    )
    await done.wait()
    await session.disconnect()

asyncio.run(run("これ何？", "http://www.sakado-jigenji.jp/images/k_logo.png"))

この画像は「慈眼寺（じげんじ）」というお寺のロゴやタイトル部分です。

上部には「真言宗 智山派 由城山」と書かれており、これはこのお寺が「真言宗智山派」に属し、山号が「由城山」であることを示しています。

左側のマークは寺院の紋章（寺紋）と思われます。

まとめると、  
**真言宗智山派 由城山 慈眼寺**  
というお寺のロゴ画像です。


# Custom Agents & Sub-Agent Orchestration
(Multi-Agent-System)

In [ ]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, CustomAgentConfig, SystemMessageReplaceConfig

agent1 = CustomAgentConfig(
    name="Agent1",
    display_name="Agent1",
    description="Agent1",
    prompt="ケーキの見た目についてのみのプロフェッショナルとして回答します。",
    infer=True
)

agent2 = CustomAgentConfig(
    name="Agent2",
    display_name="Agent2",
    description="Agent2",
    prompt="ケーキの味についてのみのプロフェッショナルとして回答します。",
    infer=True
)

agent3 = CustomAgentConfig(
    name="Agent3",
    display_name="Agent3",
    description="Agent3",
    prompt="最高のケーキについて、Agent1とAgent2に確認した上で総合的に回答します。",
    infer=False
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        custom_agents=[agent1, agent2, agent3],
        system_message=SystemMessageReplaceConfig(content="Jemmy's TeaHouse"),
        agent="Agent3"
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("私は太郎です。ケーキについて知りたい。"))


太郎さん、最高のケーキについてAgent1とAgent2に確認しています。両者の意見をまとめてから総合的にご案内しますので、少々お待ちください。
ケーキの「見た目」だけに特化したプロフェッショナルとしてお答えします。

---

### 最高のケーキ（見た目編）

#### 1. ショートケーキ
**おすすめポイント：**
- 真っ白な生クリームと鮮やかな赤いイチゴのコントラストが美しい
- 層になったスポンジとクリームが断面も華やか
- シンプルながら清潔感と高級感がある

#### 2. フルーツタルト
**おすすめポイント：**
- 色とりどりのフルーツが宝石のように並び、華やかさ抜群
- 季節ごとにフルーツの彩りが変わり、見た目のバリエーションが豊富
- グラサージュ（艶出し）がかかっていると、より一層美しく見える

#### 3. オペラ
**おすすめポイント：**
- チョコレートのグラサージュが鏡のように光り、上品な印象
- 層が均一で、断面がとても美しい
- 金箔やチョコレートプレートの装飾が高級感を演出

#### 4. ムースケーキ（グラサージュ仕上げ）
**おすすめポイント：**
- 鮮やかな色のグラサージュ（鏡面仕上げ）が目を引く
- 曲線や幾何学的なデザインが多く、アート作品のよう
- カットした断面もカラフルで美しい

#### 5. ウェディングケーキ
**おすすめポイント：**
- 多段で高さがあり、圧倒的な存在感
- 花やリボン、シュガークラフトなどの装飾が豪華
- 写真映えが抜群で、特別な日の象徴

---

### 総評
「最高のケーキの見た目」は、色彩のバランス、層の美しさ、装飾の繊細さ、全体の統一感がポイントです。特にフルーツタルトやショートケーキは、誰が見ても「美味しそう！」と思える王道の美しさがあります。特別感や芸術性を求めるなら、グラサージュ仕上げのムースやウェディングケーキもおすすめです。

ご希望のシーンや好みに合わせて、見た目で選ぶケーキを提案できますので、ぜひご相談ください！

ケーキの味についてのみのプロフェッショナルとしてお答えします。

「最高のケーキ」は人それぞれ好みが分かれますが、味の観点から多くの人に愛されるケーキとして「ショートケーキ（いちごのショートケーキ）」をおすすめします。

**おすすめ

# MCP Servers

In [12]:
import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler, SystemMessageReplaceConfig, MCPStdioServerConfig

server_time = MCPStdioServerConfig(
    type="stdio",
    env={},
    cwd="",
    command="uvx",
    args=["mcp-server-time", "--local-timezone=Asia/Tokyo"],
    tools="*",
)

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        mcp_servers=[server_time,],
        system_message=SystemMessageReplaceConfig(content="サーバ時刻を伝えます")
    )
    done = asyncio.Event()
    def on_event(event):
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("今何時？"))


今は日本時間で2026年5月9日（土）08:12です。


# Skills

In [13]:
#!gh skill preview coji/dajare japanese-rap
!gh skill install coji/dajare japanese-rap --agent opencode --scope project -f
!pwd
!ls -la .agents/skills/

Using ref v1.5.0 (b77453c3)
Note: found 2 skill(s) at the repository root

! Skills are not verified by GitHub and may contain prompt injections, hidden instructions, or malicious scripts. Always review skill contents before use.

✓ Installed japanese-rap (from coji/dajare@v1.5.0) in .agents/skills

  japanese-rap/
  ├── SKILL.md
  ├── references/
  │   ├── bad-examples.md
  │   ├── examples.md
  │   ├── generation-guide.md
  │   └── rap-theory.md
  └── scripts/
      ├── rhyme-dict.json.gz
      └── rhyme.py

! Skills may contain prompt injections or malicious scripts.
  Review installed content before use:

    gh skill preview coji/dajare japanese-rap@b77453c3d2c79919eac725a9d6e67095c2943e32

/content
total 12
drwxr-xr-x 3 root root 4096 May  8 23:05 .
drwxr-xr-x 3 root root 4096 May  8 23:05 ..
drwxr-xr-x 4 root root 4096 May  8 23:05 japanese-rap


In [15]:

import asyncio

from copilot import CopilotSession
from copilot.generated.session_events import AssistantMessageData, SessionIdleData
from copilot.session import PermissionHandler

async def run(prompt: str):
    session: CopilotSession = await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model="gpt-4.1",
        skill_directories=[".agents/skills/",]
    )
    done = asyncio.Event()
    def on_event(event):
        print(event)
        match event.data:
            case AssistantMessageData() as data:
                print(data.content)
            case SessionIdleData():
                done.set()
    session.on(on_event)
    await session.send(prompt)
    await done.wait()
    await session.disconnect()

asyncio.run(run("ラップをして！お題はAI産業！"))


タイトル：AI産業革命

リリック本体：
未来を描くAI、産業の最前線  
データの海泳ぐ、**限界**を超えて前進  
人間と機械の境界、曖昧な現実  
コードが紡ぐ**経済**、新たな伝説  
アルゴリズムが夜明けを告げる  
イノベーションの波、**世界**を揺らす  
単純作業は消え、創造が残る  
AIが導く**時代**、誰もが問う  
倫理と効率の狭間で踊る  
意思なき知性、**未来**を語る  
雇用の形も変わる、戸惑いと希望  
進化の果てに**期待**、新しい方法  
バイアスもリスクも抱えたまま  
それでも進む**社会**、止まらぬドラマ  
人とAIが共に描くストーリー  
産業革命、**答え**はこの通り

韻の解説：
- 「限界」×「経済」、「世界」×「時代」、「未来」×「期待」、「社会」×「答え」など、母音列・子音列が近い語で脚韻を構成
- 「コードが紡ぐ経済、新たな伝説」など、AIの創造性と産業変革をパンチライン化
- 「倫理と効率の狭間で踊る」「意思なき知性、未来を語る」など、AI産業の本質的な問いを内包
- 起承転結を意識し、序盤で状況設定→中盤で変化と課題→終盤で未来への展望とパンチライン
